In [ ]:
#Download zipped fasta file from NCBI and unzip it 
curl -L -O "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/005/845/GCF_000005845.2_ASM584v2/GCF_000005845.2_ASM584v2_genomic.fna.gz"
gunzip GCF_000005845.2_ASM584v2_genomic.fna.gz

#creating environment
mamba create -n project_one
conda activate project_one 

mamba create -n bacterial_genome_mapping

#download multiple packages
conda install -y -c bioconda fastqc shovill minimap2 samtools snippy bcftools abricate sra-tools
for this one i did half up including minimap2 for environment 1 not 2

#finding what environments are in conda
conda env list

#comes out with fasta files not fastq 
fasterq-dump --fasta --split-files <accession_number>

fasterq-dump --fasta --split-files SRR15197825

#comes out with fastq files 
fasterq-dump --split-files SRR41602795

# Run QC on raw reads
fastqc SRR41602795_1.fastq SRR41602795_2.fastq


#shovill assembing fastq's 
shovill --outdir ecoli_assembly --R1 SRR41602795_1.fastq --R2 SRR41602795_2.fastq --cpus 4 --ram 7  

#map onto minimap2
minimap2 -ax asm5 GCF_000005845.2_ASM584v2_genomic.fna ecoli_assembly/contigs.fasta > ecoli_assembly_ref.sam

    #use asm5 if less than 5% sequence divergence (same bacterial species) since mapping to reference genome it'll be less than 5 
    #use asm10 if more than 10% divergence 

#convert sam file to bam file 
samtools view -b -o <output_file_name>.bam <sam_file_name>.sam
 samtools view -b -o ecoli_assembly_ref.bam ecoli_assembly_ref.sam
    # -b convert to bam file
    # -o output

#code to run if ncurses error comes up
conda install -c conda-forge ncurses samtools

#sort bam file
samtools sort -o final_ecoli_assembly.bam ecoli_assembly_ref.bam

#make a .bai file for IGV
samtools index final_ecoli_assembly.bam
    #will have both a .bam and .bai file, do not delete either or rename

#check how many mapped to reference genome 
samtools flagstat final_ecoli_assembly.bam

#check to see if you have data in the file 
ls -lh contigs.fa

#find snps 
snippy [options] --outdir <dir> --ref <ref> --R1 <R1.fq.gz> --R1 <R2.fq.gz>
snippy --outdir ecoli_assembly_snps --ref GCF_000005845.2_ASM584v2_genomic.fna --R1 SRR41602795_1.fastq --R2 SRR41602795_2.fastq

#check if snps are there
samtools flagstat snps.bam

#find snps because its showing up as 0 
bcftools mpileup -f AE005174.2_1.fasta final_ecoli_assembly.bam -o raw_variants.bcf
bcftools call -mv -Ob -o snp_variants.bcf raw_variants.bcf
bcftools view snp_variants.bcf > real_snps.csv

#have to go into assembly from shovill folder 
abricate --db ncbi contigs.fasta > ecoli_resistance_results.txt
abricate --db vfdb contigs.fasta > ecoli_virulence_results.txt
    #ncbi for resistance genes and vfdb for virulence factors

: 